## encodings: PathMNIST x UNI2-h / Virchow2

Runs PathMNIST's 224px tissue patches through two pathology foundation models -- [UNI2-h](https://huggingface.co/MahmoodLab/UNI2-h) and [Virchow2](https://huggingface.co/paige-ai/Virchow2) -- and saves the resulting patch encodings to disk, one file per model. This is the fastest of the three datasets to embed (already patch-sized, no tiling needed) and a good first check that model loading + auth is working before moving on to CAMELYON17/PANDA.

Downstream, these encodings feed a probing notebook (e.g. a linear/kNN classifier on top of frozen features) -- not built here.

In [ ]:
import os

os.environ["HF_HOME"] = "/home/shared/.cache/huggingface"
os.environ["HF_HUB_CACHE"] = "/home/shared/.cache/huggingface/hub"
os.environ["TRANSFORMERS_CACHE"] = (
    "/home/shared/.cache/huggingface/hub"  # deprecated alias, harmless to set
)

In [ ]:
import sys

sys.path.append("/home/shared/helper/")
from nbhelper import (
    plt,
    np,
)

import pyrootutils

# VS Code's Jupyter kernel starts in the launch dir, not the notebook's folder,
# so an upward search from cwd can miss the repo root -- search from the notebook
# file itself (VS Code sets __vsc_ipynb_file__) and fall back to cwd otherwise.
search_from = globals().get("__vsc_ipynb_file__", ".")
root = pyrootutils.setup_root(search_from=search_from, indicator=".project-root", pythonpath=False)
sys.path.append(str(root / "src"))
from vfm_encoders import load_uni2, load_virchow2, embed_images  # noqa: E402

import torch
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm

### 0. Setup

Both encoders are gated on Hugging Face -- request access on the [UNI2-h](https://huggingface.co/MahmoodLab/UNI2-h) and [Virchow2](https://huggingface.co/paige-ai/Virchow2) model pages with the account you authenticate as below, then set `HF_TOKEN` in the environment (or leave it unset to get an interactive login prompt).

In [ ]:
import os
from huggingface_hub import login

login(token=os.environ.get("HF_TOKEN"))

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
if device == "cpu":
    print(
        "WARNING: no GPU found -- these are large ViT-H encoders, CPU is only "
        "fine for a quick smoke test on a handful of patches."
    )

### 1. Load PathMNIST

Same source as `exploration/explore_pathmnist.ipynb`. Defaults to the 1000-patch subset for a fast end-to-end run -- flip `USE_SUBSET` to `False` to embed the full 224px release once the pipeline below is confirmed working.

In [ ]:
USE_SUBSET = True

pathmnist_base_dir = "/home/shared/data/pathmnist/"
pathmnist_filename = "pathmnist_224_subset1000.npz" if USE_SUBSET else "pathmnist_224.npz"
pathmnist_path = pathmnist_base_dir + pathmnist_filename

LABEL_NAMES = {
    0: "adipose",
    1: "background",
    2: "debris",
    3: "lymphocytes",
    4: "mucus",
    5: "smooth muscle",
    6: "normal colon mucosa",
    7: "cancer-associated stroma",
    8: "colorectal adenocarcinoma epithelium",
}

data = np.load(pathmnist_path)
splits = ["train", "val", "test"]
for split in splits:
    print(f"{split:5s}: {len(data[f'{split}_labels']):>6,} patches")

### 2. Load the encoders

In [ ]:
encoders = {
    "uni2-h": load_uni2(device),
    "virchow2": load_virchow2(device),
}
for enc in encoders.values():
    n_params = sum(p.numel() for p in enc.model.parameters())
    print(f"{enc.name:10s} embed_dim={enc.embed_dim:5d}  params={n_params / 1e6:.0f}M")

### 3. Extract encodings

Encodes one split at a time, and within a split, converts + embeds patches in
small chunks rather than materializing every PIL image for the whole split (let
alone all splits) up front -- the full 224px release is large enough that doing
so exhausts memory.

In [ ]:
CHUNK_SIZE = 256  # patches converted to PIL + embedded at once, per split/encoder


def as_pil_images(image_array: np.ndarray) -> list[Image.Image]:
    return [Image.fromarray(img).convert("RGB") for img in image_array]


def encode_split_in_chunks(
    raw_images: np.ndarray, enc, device: str, chunk_size: int = CHUNK_SIZE
) -> np.ndarray:
    """Embeds a split's raw uint8 patches without ever holding more than
    `chunk_size` PIL images (and their transformed tensors) in memory at once."""
    out = np.empty((len(raw_images), enc.embed_dim), dtype=np.float32)
    chunk_starts = range(0, len(raw_images), chunk_size)
    for start in tqdm(
        chunk_starts, total=len(chunk_starts), desc=f"{enc.name} ({len(raw_images)} patches)"
    ):
        chunk = as_pil_images(raw_images[start : start + chunk_size])
        out[start : start + len(chunk)] = embed_images(
            enc, chunk, device=device, show_progress=False
        )
    return out


split_labels = {split: data[f"{split}_labels"].reshape(-1) for split in splits}

encodings = {
    enc.name: {} for enc in encoders.values()
}  # encodings[model_name][split] -> (N, embed_dim) array
for split in splits:
    raw_images = data[f"{split}_images"]
    for enc in encoders.values():
        encodings[enc.name][split] = encode_split_in_chunks(raw_images, enc, device=device)
    del raw_images

### 4. Save encodings

One `.npz` per model, keyed by split, alongside the matching labels so a later probing notebook can load features and targets together.

In [ ]:
out_dir = Path(pathmnist_base_dir) / "encodings"
out_dir.mkdir(parents=True, exist_ok=True)

suffix = "subset1000" if USE_SUBSET else "full"
for model_name, split_encodings in encodings.items():
    out_path = out_dir / f"{model_name}_{suffix}.npz"
    np.savez(
        out_path,
        **{f"{split}_encodings": split_encodings[split] for split in splits},
        **{f"{split}_labels": split_labels[split] for split in splits},
    )
    print("Saved:", out_path)

### 5. Sanity check

PCA down to 2D on the train split, colored by tissue class -- a well-trained encoder should show at least loose clustering by class before any probe is fit on top of it.

In [ ]:
from sklearn.decomposition import PCA

fig, axs = plt.subplots(1, len(encoders), figsize=(6 * len(encoders), 5))
for ax, model_name in zip(axs, encodings):
    feats = encodings[model_name]["train"]
    labels = split_labels["train"]
    coords = PCA(n_components=2, random_state=42).fit_transform(feats)

    for label, name in LABEL_NAMES.items():
        mask = labels == label
        ax.scatter(coords[mask, 0], coords[mask, 1], s=8, alpha=0.6, label=name)
    ax.set_title(model_name)

axs[-1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()